# RAG Expert Knowledge Worker

Build a question-answering assistant for **Insurellm** (Insurance Tech company) using
Retrieval-Augmented Generation.

In [ ]:
import os
import glob
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

# LangChain
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

load_dotenv(override=True)
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
print(f"OpenRouter API Key: {openrouter_api_key[:4]}...")

openai_client = OpenAI(
    api_key=openrouter_api_key,
    base_url="https://openrouter.ai/api/v1"
)

## Load & chunk the knowledge base

The Insurellm knowledge base lives in `../../knowledge-base/` and contains Markdown files
about employees, products, contracts, and company info.

In [ ]:
KB_PATH = os.path.join("..", "..", "knowledge-base")

# Load every .md file recursively
# Pass encoding="utf-8" so TextLoader can read files with non-cp1252 characters on Windows
loader = DirectoryLoader(KB_PATH, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
raw_docs = loader.load()
print(f"Loaded {len(raw_docs)} documents")

# Chunk documents into overlapping segments
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
chunks = splitter.split_documents(raw_docs)
print(f"Split into {len(chunks)} chunks")

# Preview first chunk
print("\nSample chunk:")
print(chunks[0].page_content[:300])

## Embed & store in Chroma

We use a lightweight HuggingFace sentence-transformer model to create semantic embeddings,
then persist them in a local Chroma vector database.

In [ ]:
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
DB_PATH = "./insurellm_chroma_db"

embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

# Build (or reload) the vector store
if os.path.exists(DB_PATH):
    print("Loading existing vector DB...")
    vectorstore = Chroma(
        persist_directory=DB_PATH,
        embedding_function=embeddings,
    )
else:
    print("Building vector DB from chunks...")
    vectorstore = Chroma.from_documents(
        chunks,
        embedding=embeddings,
        persist_directory=DB_PATH,
    )

print(f"Vector store contains {vectorstore._collection.count()} vectors")

## Build the RAG retriever & answer function

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

SYSTEM_PROMPT = """You are a knowledgeable assistant for Insurellm, an Insurance Technology company.
Answer questions using only the context provided. If the answer is not in the context, say so clearly.
Be concise and factual."""


def rag_answer(question: str, chat_history: list[dict]) -> str:
    """Retrieve relevant chunks and answer the question using GPT."""
    # Retrieve relevant documents
    docs = retriever.invoke(question)
    context = "\n\n---\n\n".join(doc.page_content for doc in docs)

    # Build messages with conversation history
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for turn in chat_history:
        messages.append({"role": turn["role"], "content": turn["content"]})
    messages.append({
        "role": "user",
        "content": f"Context from Insurellm knowledge base:\n{context}\n\nQuestion: {question}",
    })

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
    )
    return response.choices[0].message.content


def chat_fn(message: str, history: list[dict]):
    """Gradio chat callback — streams a single-shot response."""
    answer = rag_answer(message, history)
    yield answer

## Quick evaluation on sample questions

Test retrieval quality before launching the UI.

In [ ]:
TEST_QUESTIONS = [
    "Who is the CEO of Insurellm?",
    "What products does Insurellm offer?",
    "What is Carllm used for?",
    "How many employees does Insurellm have?",
    "What are the main features of Homellm?",
]

print("=== RAG Evaluation (sample questions) ===\n")
for q in TEST_QUESTIONS:
    docs = retriever.invoke(q)
    answer = rag_answer(q, [])
    print(f"Q: {q}")
    print(f"Retrieved {len(docs)} chunks  |  Top source: {docs[0].metadata.get('source','?').split('/')[-1]}")
    print(f"A: {answer[:200]}..." if len(answer) > 200 else f"A: {answer}")
    print()

## Launch Gradio Chat UI

In [ ]:
gr.ChatInterface(
    fn=chat_fn,
    type="messages",
    title="🏢 Insurellm Expert Knowledge Worker",
    description=(
        "Ask any question about Insurellm employees, products, contracts, or company info. "
        "Powered by RAG over the Insurellm knowledge base."
    ),
    examples=[
        ["Who is the CEO of Insurellm?"],
        ["What is Carllm? What are its features?"],
        ["Tell me about the Healthllm product"],
        ["What does Alex Chen work on?"],
        ["Compare Homellm and Carllm products"],
    ],
    theme=gr.themes.Soft(),
).launch(inbrowser=True)